# 02 — Accuracy Benchmarks (UAS / LAS)

Run both parsers on full EWT + SynTagRus test sets using gold tokenization.

In [1]:
# === Kaggle / Colab setup (skip if running locally) ===
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE or IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "../requirements.txt"], check=True)
    # Kaggle ships pandas built against numpy 2.x, but spacy 3.7.5 pins numpy<2.
    # Force-reinstall pandas+numpy together so binary ABI matches.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps",
                    "numpy<2", "pandas>=2.0,<2.3"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_trf"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "ru_core_news_lg"], check=True)
    subprocess.run([sys.executable, "../scripts/download_data.py"],
                   check=True)
    print("Setup done.")
else:
    print("Local env detected — skipping cloud setup.")


Local env detected — skipping cloud setup.


In [2]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import pandas as pd
from tqdm import tqdm

from src.data import load_sentences
from src.parsers import SpacyParser, StanzaParser
from src.metrics import uas, las, Gold, Prediction

EN = load_sentences(Path("../data/en_ewt_test.conllu"))
RU = load_sentences(Path("../data/ru_syntagrus_test.conllu"))
print(f"EN: {len(EN)} sents | RU: {len(RU)} sents")

EN: 2077 sents | RU: 8800 sents


In [3]:
CHUNK = 100

def parse_dataset(parser, sents, label):
    toks = [s.tokens for s in sents]
    parses = []
    for i in tqdm(range(0, len(toks), CHUNK), desc=label):
        parses.extend(parser.parse(toks[i:i+CHUNK]))
    preds = [Prediction(p.heads, p.deprels) for p in parses]
    golds = [Gold(s.heads, s.deprels) for s in sents]
    return uas(preds, golds), las(preds, golds), preds

rows = []
parser_configs = [
    ("en", EN, SpacyParser("en_core_web_trf")),
    ("en", EN, StanzaParser("en")),
    ("ru", RU, SpacyParser("ru_core_news_lg")),
    ("ru", RU, StanzaParser("ru")),
]

all_preds = {}
for lang, sents, parser in parser_configs:
    u, l, preds = parse_dataset(parser, sents, f"{lang}-{parser.name}")
    all_preds[(lang, parser.name)] = preds
    rows.append({"lang": lang, "parser": parser.name,
                 "family": "transition" if "spacy" in parser.name else "graph",
                 "uas": round(u, 4), "las": round(l, 4),
                 "n_sentences": len(sents)})
    print(f"{lang} {parser.name:35s}  UAS={u:.4f}  LAS={l:.4f}")

en-spacy:en_core_web_trf:   0%|          | 0/21 [00:00<?, ?it/s]/Users/aleksandrgavkovskij/programming/nlp_case_study/.venv/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
en-spacy:en_core_web_trf: 100%|██████████| 21/21 [00:19<00:00,  1.10it/s]


en spacy:en_core_web_trf                UAS=0.6152  LAS=0.4500


en-stanza:en: 100%|██████████| 21/21 [00:36<00:00,  1.74s/it]


en stanza:en                            UAS=0.9074  LAS=0.8832


ru-spacy:ru_core_news_lg: 100%|██████████| 88/88 [00:12<00:00,  6.93it/s]


ru spacy:ru_core_news_lg                UAS=0.8946  LAS=0.8446


ru-stanza:ru: 100%|██████████| 88/88 [03:09<00:00,  2.15s/it]

ru stanza:ru                            UAS=0.9358  LAS=0.9043


In [4]:
df = pd.DataFrame(rows)
df.to_csv("../results/accuracy.csv", index=False)
print("Saved results/tables/accuracy.csv")
df

Saved results/tables/accuracy.csv


,lang,parser,family,uas,las,n_sentences
0,en,spacy:en_core_web_trf,transition,0.6152,0.4500,2077
1,en,stanza:en,graph,0.9074,0.8832,2077
2,ru,spacy:ru_core_news_lg,transition,0.8946,0.8446,8800
3,ru,stanza:ru,graph,0.9358,0.9043,8800


## What we learned (from this run)

Headline numbers written to `results/accuracy.csv`:

| Lang | Parser (family) | UAS | LAS |
|---|---|---|---|
| EN | spaCy (transition) | 0.6152 | 0.4500\* |
| EN | Stanza (graph) | **0.9074** | **0.8832** |
| RU | spaCy (transition) | 0.8946 | 0.8446 |
| RU | Stanza (graph) | **0.9358** | **0.9043** |

- **Graph-based wins UAS on both languages** — +29.2 pt on EN, +4.1 pt on RU. The RU margin is the fair cross-family comparison (both parsers output UD labels there).
- \*EN LAS for spaCy is depressed by label-scheme mismatch: `en_core_web_trf` emits CLEAR-style labels (`dobj`, `pobj`, `prep`) not UD. Notebook 06 quantifies this — a label-only remap closes ~11 pt of the raw 43 pt LAS gap. Use EN **UAS** as the label-scheme-neutral accuracy metric.
- **Russian LAS gap (4 pt) is much smaller than English (39 pt)** — once labels are UD-conformant on both sides, the accuracy difference is meaningful but not dramatic. Notebook 04 shows where it concentrates (long arcs, non-projective arcs, `obl`/`flat`/`parataxis`/`compound`).
